# QNAasm

In this notebook, we will see how we can represent Quantum Algorithm that works on Netral Qtom processors using our language QNAasm.

First we include all required nodes to build a QNAasm circuit.

In [1]:
from qnaasm.nodes import (
    Block,
    Calloc,
    Conditional,
    Gate,
    Measure,
    Move,
    Qalloc,
    Qfree,
)

We can also include two utils that will help use to handle positions and classical registers.

In [2]:
from qnaasm.classical_register import ClassicalRegister
from qnaasm.position import Position

Let say we want to write the following circuit expressed in MimiQ in QNAasm.

In [3]:
from mimiqcircuits import (
    Circuit,
    GateH,
    GateCX,
    GateX,
    IfStatement,
    Measure as MimiqMeasure,
    BitString,
)

c = Circuit()
c.push(GateH(), 0)
c.push(GateCX(), 0, 1)
c.push(MimiqMeasure(), [0, 1], [0, 1])
c.push(IfStatement(GateX(), BitString("1")), 0, 0)

c.draw()

        ┌─┐   ┌──────┐                 ┌─┐                                      
 q[0]: ╶┤H├─●─┤  M   ├─────────────────┤X├─────────────────────────────────────╴
        └─┘┌┴┐└───╥──┘┌──────┐         └╥┘                                      
 q[1]: ╶───┤X├────╫───┤  M   ├──────────╫──────────────────────────────────────╴
           └─┘    ║   └───╥──┘          ║                                       
                  ║       ║             ║                                       
                  ║       ║   ┌───────┐○╝                                       
 c:    ═══════════╩═══════╩═══╪c[0]==1╪═════════════════════════════════════════
                  0       1   └───────┘                                         


To do so, we will first need to place qubits on the 2D grid of the QPU.

In [4]:
qubits = {
    0: Position(0, 0),
    1: Position(0, 5),
}

*Note: the two qubits are too far away to interact, we will have to move them to execute the CX gate.*

A QNAasm program simply is an array of QNAasm instructions. Moreover, qubits needs to be allocated and deallocated.

We can therefore write the previous circuit in QNAasm.

In [5]:
program = [
    Calloc("c", 2), # allocate a 2 bits classical register to store measures results.
    Qalloc([pos for pos in qubits.values()]), # we can allocate multiple qubits in a single instruction.
    Gate("H", [qubits[0]]),  # apply H on the first qubit.
    Block([ # we can arrange instructions into blocks, that way we can clearly see which movements are required to apply a gate.
        *[Move(Position(0, i), Position(0, i + 1)) for i in range(4)], # move first qubit near second one.
        Gate("CX", [Position(0, 4), qubits[1]]), # apply CX on the two qubits.
        *[Move(Position(0, i), Position(0, i - 1)) for i in range(4, 0, -1)], # move the first qubit to its initial position.
    ]),
    Measure([pos for pos in qubits.values()], ClassicalRegister("c")), # measure all qubits inside the register c.
    Conditional(ClassicalRegister("c", 0), 1, Gate("X", [qubits[0]])), # apply X on first qubit if c[0] is 1.
    Qfree([pos for pos in qubits.values()]), # deallocate all qubits.
]

*Note: here, we have to hardcode positions: that's because QNAasm does not manipulate qubits number but their positions directly, so that it is easier to send instructions to hardware.*

Finally, we can print the resulting program using the pretty printer.

In [6]:
from qnaasm.pretty_printer import PrettyPrinter

printer = PrettyPrinter()

for instruction in program:
    instruction.accept(printer)

calloc c[2]
qalloc {0,0},{0,5}
gate H {0,0}
{
  move {0,0} to {0,1}
  move {0,1} to {0,2}
  move {0,2} to {0,3}
  move {0,3} to {0,4}
  gate CX {0,4},{0,5}
  move {0,4} to {0,3}
  move {0,3} to {0,2}
  move {0,2} to {0,1}
  move {0,1} to {0,0}
}
measure {0,0},{0,5} to c
if c[0] = 1 then
  gate X {0,0}
qfree {0,0},{0,5}
